# GameTheory-02d — Épistémique et connaissance commune (C#)

**Navigation** : [Série GameTheory](README.md) | [Version Python](GameTheory-02d-Epistemique-Connaissance-Commune-Python.ipynb) | [Palier 02](GameTheory-02-NormalForm-Python.ipynb)

**Objectifs** : calculer connaissance, point fixe et posteriors ; éprouver Aumann. **Prérequis** : ensembles finis et probabilités conditionnelles. **Durée** : 2 h.

## Pourquoi ce carnet

Le palier 02 élimine les stratégies dominées et énumère les supports mixtes. Il le fait sans jamais dire **ce qu'un joueur sait**. Or l'élimination itérée n'est pas une règle de calcul qu'on postule : c'est la conséquence d'une hypothèse sur la connaissance des joueurs, et cette hypothèse a un nom précis — la **connaissance commune de la rationalité**.

Ce carnet construit la machinerie minimale qui rend cette phrase vérifiable par le calcul :

1. un **modèle à partitions** — un ensemble de mondes possibles, une partition par joueur, un opérateur de connaissance ;
2. la **connaissance commune** comme point fixe, distincte de la connaissance mutuelle ;
3. le théorème d'**agreeing to disagree** d'Aumann (1976), éprouvé ici sur des modèles finis par énumération exhaustive, avec son contrôle négatif.

Le fil directeur est un piège de formalisation que le carnet mesure au lieu de l'éviter : « les joueurs sont en désaccord » et « les joueurs savent qu'ils sont en désaccord **sur ces valeurs-là** » sont deux événements différents, et seul le second est impossible sous prior commun.

## 1. Un modèle épistémique : mondes, partitions, connaissance

On se donne un ensemble fini de **mondes** $\Omega$ — chaque monde est une description complète de l'état des choses, y compris ce que chacun sait. L'information du joueur $i$ est une **partition** $P_i$ de $\Omega$ : dans le monde $\omega$, le joueur $i$ ne distingue pas $\omega$ des autres mondes de la même cellule $P_i(\omega)$.

L'**opérateur de connaissance** est alors entièrement déterminé par l'information :

$$K_i(E) = \{\omega \in \Omega : P_i(\omega) \subseteq E\}$$

Un joueur *sait* $E$ quand son information, quelle que soit la vraie situation, ne laisse aucune place à $\lnot E$. La connaissance n'est pas une croyance : elle est vraie par construction ($K_i(E) \subseteq E$), et c'est ce qui lui donne les trois axiomes dits **S5** — véracité, introspection positive ($K_i E \Rightarrow K_i K_i E$) et introspection négative ($\lnot K_i E \Rightarrow K_i \lnot K_i E$).

En C#, `Prior` contient des poids entiers strictement positifs, éventuellement non normalisés ; `Post` les normalise en divisant la masse de l’événement dans la cellule par la masse de cette cellule.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;
record Model(int N, int[][] Parts, int[] Prior) {
 public int Full => (1 << N)-1;
 public int Cell(int i,int w) => Parts[i].Single(b => (b & (1<<w)) != 0);
 public virtual int Know(int i,int e) => Enumerable.Range(0,N).Where(w => (Cell(i,w)&~e)==0).Sum(w=>1<<w);
 public bool Valid() => Parts.All(p => p.All(b=>b!=0 && (b&~Full)==0) && p.Sum(b=>BitOperations.PopCount((uint)b))==N && p.Aggregate(0,(a,b)=>a|b)==Full);
}
record ProbeS5() : Model(2,new[]{new[]{3}},new[]{1,1}) {
 public override int Know(int i,int e) => 1;
}
string Bits(int mask,int n=4) => "{"+string.Join(",",Enumerable.Range(0,n).Where(i=>(mask&(1<<i))!=0))+"}";
var model = new Model(4, new[]{new[]{3,12},new[]{1,6,8}},new[]{1,1,1,1});
int E=7;
Console.WriteLine($".NET {Environment.Version} ; modèle valide : {model.Valid()}");
Console.WriteLine($"K0(E)={Bits(model.Know(0,E))} ; K1(E)={Bits(model.Know(1,E))}");

.NET 10.0.14 ; modèle valide : True


K0(E)={0,1} ; K1(E)={0,1,2}


### Lecture du résultat

`K_0(E) = {0, 1}` : quand le monde vaut 0 ou 1, le joueur 0 sait que $E$ — sa cellule $\{0,1\}$ est entièrement contenue dans $E$. Aux mondes 2 et 3, sa cellule $\{2,3\}$ déborde sur le 3, qui n'est pas dans $E$ : il ne sait pas.

`K_1(E) = {0, 1, 2}` : le joueur 1 est mieux informé sur cette partie du monde, il sait $E$ partout sauf au monde 3.

Les deux opérateurs diffèrent, et c'est tout l'objet du carnet : **la connaissance dépend de l'information, pas du monde seul**. Un même événement peut être connu d'un joueur et ignoré de l'autre au même monde.

### Exercice 1 — vérifier les axiomes S5 sur un modèle que vous construisez

Les trois axiomes S5 se vérifient par énumération exhaustive : pour tout monde et tout événement, les trois inclusions doivent tenir. Écrivez la fonction `CheckS5(model)` qui renvoie un dictionnaire `{axiome: bool}` pour `"veracite"`, `"intro_positive"`, `"intro_negative"`.

Deux pièges, tous deux dans la définition et non dans le code : (a) la véracité porte sur $K_i(E) \subseteq E$, pas sur $E \subseteq K_i(E)$ ; (b) l'introspection négative demande un événement qui est lui-même un ensemble de mondes — $\lnot K_i E$ est $\Omega \setminus K_i(E)$, pas une proposition sur la partition.

La garde teste aussi une sonde artificielle non factive : elle ne dérive pas d’une partition. Elle vérifie que votre fonction sait détecter une violation et ne renvoie pas simplement trois valeurs vraies.

In [2]:
Dictionary<string,bool> CheckS5(Model m) {
 // TODO : parcourir tous les événements et joueurs, calculer les trois propriétés.
 // Étape 1 : vérifier K(E) inclus dans E.
 // Étape 2 : vérifier K(E) inclus dans K(K(E)).
 // Étape 3 : vérifier complément de K(E) inclus dans K(complément de K(E)).
 return null;
}
bool ValidS5Answer(Func<Model,Dictionary<string,bool>> f) {
 var keys=new[]{"veracite","intro_positive","intro_negative"};
 foreach(var test in new[]{(m:model, expected:new[]{true,true,true}),(m:(Model)new ProbeS5(),expected:new[]{false,true,false})}) {
  var r=f(test.m); if(r==null || r.Count!=3 || !keys.All(r.ContainsKey)) return false;
  if(!keys.Select(k=>r[k]).SequenceEqual(test.expected))return false;
 } return true;
}
var answerS5 = CheckS5(model);
Console.WriteLine(answerS5 == null ? "Exercice à compléter : check S5" : $"Format exact : {ValidS5Answer(CheckS5)}");

Exercice à compléter : check S5


### Les six propriétés S5, mesurées par énumération exhaustive

L'exercice 1 vous laisse construire votre propre vérification ; voici celle du carnet, étendue aux **six** propriétés de l'opérateur de connaissance que Bonanno énumère (exercice 7.5) : **Véracité** ($K_iE \subseteq E$), **Cohérence** ($K_iE \cap K_i\neg E = \emptyset$), **Introspection positive** ($K_iE \subseteq K_iK_iE$), **Introspection négative** ($\neg K_iE \subseteq K_i\neg K_iE$), **Monotonie** ($E \subseteq F \Rightarrow K_iE \subseteq K_iF$) et **Conjonction** ($K_iE \cap K_iF = K_i(E \cap F)$).

Sur le monde témoin, l'ensemble des événements est fini — $2^4 = 16$ — et chaque propriété se vérifie sur chacun d'eux, pour chaque joueur, en comparant des ensembles. Monotonie et Conjonction croisent chaque événement avec chaque autre : $16^2 = 256$ paires. Un échec incrémenterait un compteur ; la sortie ne doit montrer que des zéros.

In [3]:
for(int player=0;player<2;player++) {
 var v=new int[6];
 for(int e=0;e<16;e++) { int k=model.Know(player,e), nk=15^k;
  if((k&~e)!=0)v[0]++; if((k&model.Know(player,15^e))!=0)v[1]++;
  if((k&~model.Know(player,k))!=0)v[2]++; if((nk&~model.Know(player,nk))!=0)v[3]++;
  for(int f=0;f<16;f++) {if((e&~f)==0 && (k&~model.Know(player,f))!=0)v[4]++;
   if((k&model.Know(player,f))!=model.Know(player,e&f))v[5]++;}
 }
 Console.WriteLine($"Joueur {player}, 16 événements et 256 couples : {string.Join(", ",v)} violations (véracité, cohérence, introspections, monotonie, conjonction)");
}

Joueur 0, 16 événements et 256 couples : 0, 0, 0, 0, 0, 0 violations (véracité, cohérence, introspections, monotonie, conjonction)


Joueur 1, 16 événements et 256 couples : 0, 0, 0, 0, 0, 0 violations (véracité, cohérence, introspections, monotonie, conjonction)


### Lecture du résultat

Zéro violation, sur 16 événements par joueur et 256 paires pour Monotonie et Conjonction : les six propriétés ne sont pas des hypothèses ajoutées au modèle — elles découlent de la définition de $K_i(E)$ par la partition. Aucun modèle à partitions ne peut en faire échouer une ; l'exercice 1, sur un modèle de votre cru, doit retrouver ces zéros.

## 2. Connaissance mutuelle ou connaissance commune ?

Deux notions que le langage courant confond et que le calcul sépare :

- la **connaissance mutuelle** de $E$ : tout le monde sait $E$ — c'est $M(E) = \bigcap_i K_i(E)$ ;
- la **connaissance commune** de $E$ : tout le monde sait $E$, tout le monde sait que tout le monde sait $E$, et ainsi de suite — c'est le **plus grand point fixe** de l'opérateur $E \mapsto M(E) \cap E$.

Sur un monde fini, la suite $E_0 = E$, $E_{n+1} = M(E_n) \cap E_n$ est décroissante : elle se stabilise en au plus $|\Omega|$ étapes, et sa limite **est** $CK(E)$. Aucune infinité n'est nécessaire pour la calculer.

In [4]:
int Mutual(Model m,int e) => Enumerable.Range(0,m.Parts.Length).Aggregate(m.Full,(a,i)=>a&m.Know(i,e));
(int,List<int>) Common(Model m,int e) { var trace=new List<int>{e}; while(true){int next=Mutual(m,e)&e; trace.Add(next); if(next==e)return(e,trace); e=next;} }
var ck=Common(model,E);
Console.WriteLine($"M(E)={Bits(Mutual(model,E))} ; trace={string.Join(" -> ",ck.Item2.Select(x=>Bits(x)))}");
Console.WriteLine($"K0 K1(E)={Bits(model.Know(0,model.Know(1,E)))} ; K1 K0(E)={Bits(model.Know(1,model.Know(0,E)))}");

M(E)={0,1} ; trace={0,1,2} -> {0,1} -> {0} -> {} -> {}


K0 K1(E)={0,1} ; K1 K0(E)={0}


### Lecture du résultat

La suite des itérés est `{0,1,2} -> {0,1} -> {0} -> {} -> {}` : elle se stabilise sur l'ensemble **vide** en trois pas. La connaissance commune de $E$ est donc vide — personne, dans aucun monde, ne sait que $E$ est de connaissance commune.

Et pourtant la connaissance mutuelle **n'est pas vide** : `M(E) = {0, 1}`. Aux mondes 0 et 1, les deux joueurs savent $E$. On peut même empiler un cran : `K_0 K_1(E) = {0, 1}` et `K_1 K_0(E) = {0}` — un niveau de raisonnement de plus tient encore.

C'est la distinction que le carnet veut rendre visible, et elle est **plus forte** que ce que la prose suggère d'habitude : la connaissance mutuelle, et même quelques niveaux de « je sais qu'il sait », peuvent tenir alors que la connaissance commune est vide. Le passage à l'infini n'est pas une formalité — il retire des mondes à chaque cran.

### Le meet des partitions — l'autre visage de la connaissance commune

La caractérisation duale (Bonanno, section 7.3) : $E$ est connaissance commune en $\omega$ si et seulement si la **cellule du meet** des partitions qui contient $\omega$ est tout entière incluse dans $E$. Le meet est la partition la plus fine qui soit à la fois moins fine que celle du joueur 1 et que celle du joueur 2 : deux mondes y partagent une cellule dès que l'on peut passer de l'un à l'autre en enchaînant « être dans la même cellule d'un joueur ». Elle se calcule par fermeture transitive — une fermeture transitive des blocs.

L'équivalence des deux définitions se mesure : pour chacun des 16 événements et chacun des 4 mondes — 64 couples — l'appartenance à $CK(E)$ doit coïncider avec l'inclusion de la cellule du meet dans $E$, sans un seul contre-exemple.

In [5]:
int[] Meet(int n,params int[][] parts) { var remaining=(1<<n)-1; var result=new List<int>();
 while(remaining!=0){int c=remaining&-remaining,old; do {old=c; foreach(var part in parts) foreach(int b in part) if((b&c)!=0)c|=b;}while(c!=old); result.Add(c);remaining &= ~c;} return result.ToArray(); }
var meet=Meet(4,model.Parts);
int errors=0; for(int e=0;e<16;e++) for(int w=0;w<4;w++) if(((Common(model,e).Item1&(1<<w))!=0)!=((meet.Single(b=>(b&(1<<w))!=0)&~e)==0))errors++;
Console.WriteLine($"Meet : {string.Join(" | ",meet.Select(x=>Bits(x)))} ; 64 couples, {errors} contre-exemples");

Meet : {0,1,2,3} ; 64 couples, 0 contre-exemples


### Lecture du résultat

Aucun contre-exemple sur les 64 couples : l'itération jusqu'au point fixe et la lecture sur la cellule du meet désignent exactement les mêmes mondes. La connaissance commune n'est pas seulement une limite d'itération — c'est une propriété **locale** : elle tient en $\omega$ dès que le morceau du meet de $\omega$ tient dans $E$. C'est cette lecture locale qui rend le théorème d'Aumann manipulable dans la section suivante.

### Exercice 2 — trouver un modèle où la connaissance commune n'est pas vide

Construisez un modèle à trois mondes, au moins deux joueurs et un prior strictement positif et un événement $E$ tels que $CK(E) = E$ et $\emptyset \neq E \subsetneq \Omega$. Autrement dit : un événement qui est de connaissance commune sans être trivial.

Indice : il faut que $E$ soit une **réunion de cellules pour chaque joueur** — c'est exactement la condition d'auto-évidence (*self-evident event*) qui caractérise la connaissance commune. Vérifiez votre construction avec `Common`, puis expliquez en une phrase pourquoi l'événement trivial $\Omega$ est toujours de connaissance commune et pourquoi ce n'est pas un cas intéressant.

In [6]:
(Model m,int? e) BuildCommon() { // TODO : trois mondes, événement propre non vide de connaissance commune
 return (null,null);
}
bool ValidCommon((Model m,int? e) x) => x.m != null && x.m.N==3 && x.m.Parts.Length>=2 && x.m.Valid() && x.m.Prior.Length==3 && x.m.Prior.All(p=>p>0) && x.e.HasValue && x.e.Value>0 && x.e.Value<x.m.Full && Common(x.m,x.e.Value).Item1==x.e.Value;
var exercise2=BuildCommon(); Console.WriteLine(exercise2.m==null ? "Exercice à compléter : construire le modèle" : $"Contrat vérifié : {ValidCommon(exercise2)}");

Exercice à compléter : construire le modèle


## 3. Agreeing to disagree — et le piège de la formalisation

Deux agents partagent un **prior commun** $\mu$ sur $\Omega$ et ont chacun leur information $P_1, P_2$. Le posterior du joueur $i$ au monde $\omega$ est

$$q_i(\omega) = \mu\big(E \mid P_i(\omega)\big)$$

**Théorème d'Aumann (1976).** Si les deux agents ont le même prior, il n'existe aucun monde où il soit de connaissance commune que $q_1 = a$ et $q_2 = b$ avec $a \neq b$.

Le carnet ne se contente pas de citer le théorème : il l'**éprouve par énumération exhaustive** sur tous les modèles à quatre mondes — les 15 partitions de $\Omega$, leurs 225 couples, et les 14 événements non triviaux. Soit 3150 modèles-événements.

La formalisation du test est le point délicat. « Ils sont en désaccord » est l'événement $D = \{\omega : q_1(\omega) \neq q_2(\omega)\}$. Ce n'est **pas** l'énoncé du théorème, qui porte sur la connaissance commune des **valeurs** : pour chaque couple $(a, b)$ avec $a \neq b$, l'événement $V_{a,b} = \{\omega : q_1(\omega) = a,\ q_2(\omega) = b\}$. Le code teste $V_{a,b}$, et la cellule suivante mesure aussi ce que donnerait le test naïf sur $D$.

In [7]:
IEnumerable<int[]> Partitions(int n) { var a=new List<int>(); IEnumerable<int[]> Rec(int i) { if(i==n){yield return a.ToArray();yield break;} for(int j=0;j<a.Count;j++){a[j]|=1<<i;foreach(var p in Rec(i+1))yield return p;a[j]^=1<<i;} a.Add(1<<i);foreach(var p in Rec(i+1))yield return p;a.RemoveAt(a.Count-1); } return Rec(0); }
(int,int) Ratio(int a,int b) {int x=a,y=b;while(y!=0){int t=x%y;x=y;y=t;} return (a/x,b/x);}
(int,int) Post(Model m,int player,int ev,int world) {int b=m.Cell(player,world);return Ratio(Enumerable.Range(0,m.N).Where(i=>(b&ev&(1<<i))!=0).Sum(i=>m.Prior[i]),Enumerable.Range(0,m.N).Where(i=>(b&(1<<i))!=0).Sum(i=>m.Prior[i]));}
var ps4=Partitions(4).ToArray(); int pairs=0,viol=0,naive=0;
foreach(var a in ps4)foreach(var b in ps4){var m=new Model(4,new[]{a,b},new[]{1,1,1,1});for(int ev=1;ev<15;ev++){
 var qa=Enumerable.Range(0,4).Select(w=>Post(m,0,ev,w)).ToArray(); var qb=Enumerable.Range(0,4).Select(w=>Post(m,1,ev,w)).ToArray();
 foreach(var q in Enumerable.Range(0,4).Select(w=>(qa[w],qb[w])).Distinct().Where(q=>q.Item1!=q.Item2)) {int values=Enumerable.Range(0,4).Where(w=>qa[w]==q.Item1 && qb[w]==q.Item2).Sum(w=>1<<w);pairs++;if(Common(m,values).Item1!=0)viol++;}
 int d=Enumerable.Range(0,4).Where(w=>qa[w]!=qb[w]).Sum(w=>1<<w);naive+=BitOperations.PopCount((uint)Common(m,d).Item1);
}}
Console.WriteLine($"{ps4.Length} partitions, {ps4.Length*ps4.Length*14} modèles-événements ; {pairs} paires de valeurs, {viol} violations ; test naïf : {naive}");

15 partitions, 3150 modèles-événements ; 5632 paires de valeurs, 0 violations ; test naïf : 4832


### Lecture du résultat

Le test correct rend **0 violation** sur les 5632 paires de valeurs en désaccord examinées : sous prior commun, aucun couple de posteriors distincts n'est jamais de connaissance commune. Le théorème tient sur l'énumération complète, pas sur un exemple choisi.

Le test naïf, lui, rend **4832 cas** de désaccord de connaissance commune. Ce n'est pas une réfutation d'Aumann : c'est la mesure de l'écart entre deux événements différents. Que les agents sachent qu'ils **ne sont pas d'accord** n'empêche rien — ils peuvent parfaitement le savoir, et même le savoir mutuellement, dans un modèle où chacun interprète l'information différemment. Ce que le théorème interdit, c'est qu'ils sachent **sur quelles valeurs précises** ils diffèrent, parce que cette connaissance-là, jointe au prior commun, permettrait de recalculer la moyenne des deux posteriors et de forcer l'égalité.

Un carnet qui testerait $D$ conclurait « Aumann est faux » et aurait tort. La formalisation est ici tout le contenu, et le nombre 4832 est là pour qu'on s'en souvienne.

In [8]:
var m1=new Model(4,new[]{new[]{15},new[]{15}},new[]{1,1,1,1}); var m2=m1 with {Prior=new[]{7,1,1,1}};
Console.WriteLine($"Priors distincts, partitions grossières : q1={Post(m1,0,1,0)} q2={Post(m2,1,1,0)} ; CK(désaccord)={Bits(Common(m1,15).Item1)}");

Priors distincts, partitions grossières : q1=(1, 4) q2=(7, 10) ; CK(désaccord)={0,1,2,3}


### Lecture du contrôle négatif

Avec le même prior, deux agents qui ont cette même information calculent le même posterior : pas de désaccord. Avec des priors différents, l'écart existe et il est **de connaissance commune partout** — les deux partitions sont grossières ; ici, le désaccord est l’événement Ω, qui est auto-évident.

C'est exactement ce que le théorème autorise : il ne dit pas que le désaccord est impossible, il dit que **le prior commun l'interdit**. Retirer l'hypothèse du prior commun suffit à faire apparaître le désaccord de connaissance commune. Le contrôle négatif n'est donc pas décoratif : sans lui, un carnet qui ne trouverait aucun désaccord ne saurait pas distinguer « le théorème tient » de « le modèle est dégénéré ».

### Plus loin : le monde à six états — 203 partitions, 41 209 couples

Le banc à quatre mondes balayait 15 partitions et 225 couples. Le monde à six états change l'ordre de grandeur : le nombre de Bell $B_6 = 203$ donne $203^2 = 41\,209$ couples d'informations, croisés avec les $2^6 = 64$ événements. Deux adaptations rendent le balayage exact et supportable :

- les mondes deviennent des **masques binaires** et les partitions des tuples de masques — le meet se calcule par fermeture transitive sur six états ;
- les posteriors sont comparés en **rationnels exacts** — chaque rapport $a/b$ réduit par pgcd — pour qu'aucun arrondi flottant ne fabrique ni ne masque une égalité.

Le critère de connaissance commune devient : les deux posteriors sont constants sur la cellule du meet (chacun connaît son posterior partout dans sa cellule ; la connaissance commune exige la constance sur toute la cellule publique). Chaque cas ainsi isolé doit vérifier l'égalité — une seule égalité fausse réfuterait le théorème sur ce monde.

In [9]:
var ps6=Partitions(6).ToArray();
(long cases,long violations,string example) Scan(int[] p1,int[] p2) {
 long cases=0,violations=0;string example=""; int[] weights1=new int[64],weights2=new int[64];
 for(int mask=0;mask<64;mask++)for(int w=0;w<6;w++)if((mask&(1<<w))!=0){weights1[mask]+=p1[w];weights2[mask]+=p2[w];}
 foreach(var a in ps6)foreach(var b in ps6)foreach(int meetCell in Meet(6,a,b))for(int ev=0;ev<64;ev++) {
  var aa=a.Where(x=>(x&meetCell)!=0).ToArray();var bb=b.Where(x=>(x&meetCell)!=0).ToArray();
  var q1=Ratio(weights1[ev&aa[0]],weights1[aa[0]]); if(aa.Any(x=>Ratio(weights1[ev&x],weights1[x])!=q1))continue;
  var q2=Ratio(weights2[ev&bb[0]],weights2[bb[0]]); if(bb.Any(x=>Ratio(weights2[ev&x],weights2[x])!=q2))continue;
  cases++;if(q1!=q2){violations++;if(example=="")example=$"P1={string.Join(" | ",a.Select(x=>Bits(x,6)))} ; P2={string.Join(" | ",b.Select(x=>Bits(x,6)))} ; meet={Bits(meetCell,6)}, E={Bits(ev,6)}, q1={q1}, q2={q2}";}
 }
 return(cases,violations,example);
}
Console.WriteLine($"{ps6.Length} partitions, {ps6.Length*ps6.Length} couples, 64 événements");
var common6=Scan(new[]{1,1,1,1,1,1},new[]{1,1,1,1,1,1});
Console.WriteLine($"Prior commun : {common6.cases} cas, {common6.violations} désaccords");

203 partitions, 41209 couples, 64 événements


Prior commun : 1760900 cas, 0 désaccords


### Lecture du résultat

$1\,760\,900$ cas de posteriors connaissance commune, et pas un seul désaccord : à l'échelle du monde à six états, l'énoncé d'Aumann tient partout où ses hypothèses tiennent. Le banc à quatre mondes (3150 modèles-événements) et celui-ci explorent le même théorème à deux échelles : le petit isole la mécanique des valeurs $V_{a,b}$ et le piège du test naïf, le grand mesure la frontière.

### La frontière du théorème, mesurée au monde à six états


In [10]:
var distinct6=Scan(new[]{1,1,1,1,1,1},new[]{1,1,1,1,1,4});
Console.WriteLine($"Priors distincts : {distinct6.cases} cas, {distinct6.violations} désaccords ; exemple {distinct6.example}");

Priors distincts : 1753520 cas, 53962 désaccords ; exemple P1={0,1,2,3,4,5} ; P2={0,1,2,3,4,5} ; meet={0,1,2,3,4,5}, E={0}, q1=(1, 6), q2=(1, 9)


### Lecture du contrôle négatif

L'hypothèse « same priors » n'était pas décorative — le contrôle négatif à quatre mondes le laissait deviner sur des partitions triviales. Ici, le même balayage est rejoué, mais le joueur 2 part d'un prior tenté $(1,1,1,1,1,4)$ : le critère de constance sur le meet isole encore des cas de posteriors connaissance commune, et cette fois l'égalité a cédé — la sortie montre les deux visages, le décompte puis un cas concret.

$53\,962$ désaccords de connaissance commune sur $1\,753\,520$ cas : à peine 3 % des cas isolés, mais ils existent — et le cas affiché montre la mécanique. Les deux joueurs ont des posteriors constants sur la cellule du meet (les valeurs exactes sont affichées dans la sortie), donc de connaissance commune, et néanmoins différents : chacun conditionne juste, mais sur des croyances antérieures incompatibles. La frontière du théorème n'est pas l'information — elle est le prior.

### Exercice 3 — trois agents, même théorème

Le théorème d'Aumann s'énonce pour deux agents ; avec trois agents qui partagent un prior, la conclusion tient encore. Vérifiez-le sur un seul modèle, sans balayage exhaustif : retournez `(model, event)` : construisez trois partitions de $\{0,1,2,3\}$ et un événement $E$, calculez les trois posteriors, et cherchez un monde où les trois valeurs sont deux à deux distinctes. Montrez que l'événement « ces trois valeurs précises » n'est jamais de connaissance commune.

Indice : `Common` prend un modèle ; `Model` accepte autant de partitions que vous voulez dans son tuple. Cherchez d'abord un désaccord entre trois agents, puis testez sa connaissance commune — le résultat intéressant est que le désaccord existe, abondamment, mais qu'il n'est jamais su.

In [11]:
(Model m,int? e) ThreeAgents() { // TODO : trois partitions, un événement et trois posteriors distincts dans un monde
 return (null,null);
}
bool ValidThree((Model m,int? e) x) { if(x.m==null || x.m.N!=4 || x.m.Parts.Length!=3 || !x.m.Valid() || !x.e.HasValue || x.e<1 || x.e>=15 || x.m.Prior.Length!=4 || x.m.Prior.Any(p=>p<=0))return false;
 var m=x.m;int e=x.e.Value; for(int w=0;w<4;w++){var q=Enumerable.Range(0,3).Select(i=>Post(m,i,e,w)).ToArray();if(q.Distinct().Count()!=3)continue;
 int v=Enumerable.Range(0,4).Where(z=>Enumerable.Range(0,3).All(i=>Post(m,i,e,z)==q[i])).Sum(z=>1<<z);if(Common(m,v).Item1==0)return true;}return false; }
var ex3=ThreeAgents();Console.WriteLine(ex3.m==null?"Exercice à compléter : trois agents":"Contrat vérifié : "+ValidThree(ex3));

Exercice à compléter : trois agents


## 4. Ce que l'épistémique change pour l'élimination itérée

Reste à relier tout cela au palier 02. La rationalité d'un joueur est un événement du modèle : au monde $\omega$, le joueur $i$ est rationnel si sa stratégie maximise son espérance conditionnelle à sa cellule d'information. L'hypothèse **CKR** — *common knowledge of rationality* — est alors $CK(R)$ avec $R = \bigcap_i R_i$, et elle a une conséquence calculable sur les stratégies.

Sur un jeu fini, l'élimination itérée des stratégies strictement dominées (IESDS) illustre une conséquence de la connaissance commune de la rationalité sous les hypothèses épistémiques appropriées : chaque tour correspond intuitivement à un niveau supplémentaire de raisonnement. Le code ci-dessous fait tourner l'élimination sur un jeu $3 \times 2$ et affiche l'ensemble résiduel — ce calcul seul ne construit pas un modèle épistémique de rationalité ; il illustre la cascade motivée par cette hypothèse.

In [12]:
int[,,] game={{{3,3},{1,1}},{{2,2},{1,1}},{{0,0},{0,0}}};
var rows=new List<int>{0,1,2};var cols=new List<int>{0,1};int rounds=0;
while(true){var rr=rows.Where(r=>rows.Any(s=>s!=r && cols.All(c=>game[s,c,0]>game[r,c,0]))).ToArray();var cc=cols.Where(c=>cols.Any(d=>d!=c && rows.All(r=>game[r,d,1]>game[r,c,1]))).ToArray();if(rr.Length+cc.Length==0)break;rounds++;Console.WriteLine($"Tour {rounds} : lignes {string.Join(",",rr.Select(x=>"TMB"[x]))} ; colonnes {string.Join(",",cc.Select(x=>"LR"[x]))}");rows.RemoveAll(rr.Contains);cols.RemoveAll(cc.Contains);}
Console.WriteLine($"Survivants : {string.Join(",",rows.Select(x=>"TMB"[x]))} / {string.Join(",",cols.Select(x=>"LR"[x]))}");

Tour 1 : lignes B ; colonnes 


Tour 2 : lignes  ; colonnes R


Tour 3 : lignes M ; colonnes 


Survivants : T / L


### Lecture du résultat

L'élimination retire `['B']`, puis `['R']`, puis `['M']` : trois tours, et l'ensemble survivant est le singleton `(['T'], ['L'])`. La cascade est instructive. Au premier tour, `B` est strictement dominée par `M` (2 > 0 et 1 > 0). Au deuxième, `R` devient strictement dominée par `L` — mais elle ne l'était **pas** avant, parce que sur la ligne `B` les deux colonnes rapportaient 0 : la dominance n'apparaît qu'après une première élimination. Au troisième tour, `M` tombe face à `T`. C'est cette dépendance en chaîne qui fait de l'élimination un calcul **itératif** et non une simple passe, et c'est elle que la connaissance commune de la rationalité autorise : chaque tour suppose que l'adversaire a déjà raisonné le tour précédent.

Ce qui compte pour ce carnet n'est pas le résultat du jeu mais la **forme** du calcul : chaque tour d'élimination suppose que le joueur sait que son adversaire est rationnel, donc retire une stratégie, ce qui rend une nouvelle stratégie dominée au tour suivant. C'est la suite $M^n$ de la section 2 appliquée à l'événement de rationalité. La connaissance commune n'est donc pas un ornement philosophique du palier 02 : c'est ce qui donne un sens à « éliminer itérativement » plutôt qu'à « éliminer une fois ».

## Synthèse

Trois choses ont été **calculées** ici, et non citées :

| Résultat | Comment il est obtenu | Ce que le carnet ajoute |
|---|---|---|
| Les axiomes S5 tiennent sur un modèle à partitions | énumération exhaustive : 16 événements, 256 paires, six propriétés | la connaissance est dérivée de l'information, pas postulée |
| La connaissance mutuelle peut tenir alors que la connaissance commune est vide | itération jusqu'au point fixe, avec la suite des itérés affichée | la distinction n'est pas graduelle, elle est de nature |
| La connaissance commune se lit sur la cellule du meet | fermeture transitive des partitions, comparaison des deux définitions couple par couple | 64 couples (événement, monde), 0 contre-exemple |
| Sous prior commun, un désaccord de posteriors n'est jamais de connaissance commune | énumération exhaustive : 15 partitions, 225 couples, 14 événements | 5632 paires examinées, 0 violation — et 4832 faux contre-exemples si l'on teste la mauvaise formalisation ; au monde à six états : $1\,760\,900$ cas de posteriors connaissance commune, 0 désaccord, $53\,962$ dès que les priors divergent |

## Suite

Les chapitres suivants de Bonanno (10 à 12 : équilibre séquentiel faible, équilibre séquentiel, équilibre bayésien parfait) ne sont pas traités ici. Le palier 12 présente l'équilibre bayésien parfait en application, sans le dériver de la machinerie épistémique construite ici. Ce sont ces chapitres qui l'utilisent : les **croyances hors équilibre** sont exactement des cellules d'information auxquelles on affecte un posterior, et la cohérence d'une évaluation est une condition sur la façon dont ces posteriors se déduisent les uns des autres.

## Références

- Bonanno, G. (2015). *Game Theory* — ch. 7 *Knowledge and Common Knowledge*, ch. 8 *Beliefs*, ch. 9 *Common Knowledge of Rationality*. Licence CC BY-NC-ND 4.0 (page de copyright du PDF archivé). Copie archivée au gisement bibliographique (dossier `GameTheory`).
- Aumann, R. J. (1976). « Agreeing to Disagree ». *The Annals of Statistics* 4(6), 1236-1239 — l'énoncé du théorème et sa démonstration par martingales. Copie archivée au gisement bibliographique (dossier `GameTheory`) ; énoncé original lu p. 1236.
- Geanakoplos, J. (1992). « Common Knowledge ». *Journal of Economic Perspectives* 6(4), 53-82 — la revue de référence sur les modèles à partitions et leurs limites.
- Halpern, J. Y. & Moses, Y. (1992). « A Guide to Completeness and Complexity for Modal Logics of Knowledge and Belief ». *Artificial Intelligence* 54, 319-379 — la variante probabiliste de S5 utilisée ici.

## À retenir
Les partitions définissent la connaissance ; le meet permet de vérifier la connaissance commune. Les priors distincts montrent la portée de l’hypothèse d’Aumann.

## Vérifiez votre compréhension
Pourquoi un désaccord connu ne contredit-il pas Aumann ?
<details><summary>Réponse</summary>Le théorème porte sur la connaissance commune des valeurs précises des posteriors.</details>

## Pour aller plus loin
[Série GameTheory](README.md) et Aumann (1976), Agreeing to Disagree.